# Catching and Tracing Exceptions

In this lesson, you will learn to handle failures during program execution and trace which statements and method calls they interrupt.

CSC-239 · Module 7 · Lesson 1 of 4

Your earlier programs used return values to carry successful results from methods to callers. A failed operation needs a different path. You will follow that path through a small packing calculator, then decide where a handler belongs so one failed request does not discard later requests.

Use the [Module 7 glossary](terms.md) to revisit the vocabulary after reading each explanation.

## Learning Goals

- Trace successful and failed paths through a `try`/`catch` statement and explain which statements are skipped.
- Place a handler inside a loop so each independent input receives a response.
- Use an exception message and the first recorded method calls to locate a failure's source.

## Why This Matters

A method cannot always deliver the result its caller requested. A calculation may receive an invalid value, and later in this course a file operation may fail because a file cannot be opened. Exception handling gives a program a defined response to a failure. It also lets the program continue with independent work when doing so is appropriate.

The location of that response matters. A handler around an entire batch can discard work that was still possible for later entries. A handler around one entry can reject that entry and continue. Diagnostic information serves a different purpose: it helps a developer locate the failed operation instead of guessing from the last visible output. You will use these distinctions when building methods with explicit failure rules and testing their behavior in the next lessons.

## Check Your Starting Point

The following program contains only ideas from earlier modules. Before running anything, write its two output lines and explain why each `println` waits for `share` to return a value.

```java
class StartingShare {
    public static int share(int total, int groups) {
        return total / groups;
    }
}
for (int groups : new int[] {3, 4}) {
    System.out.println("Share: " + StartingShare.share(12, groups));
}
```

Use whole-number division and follow the array's order. This is a reading check; the snippet is shown for inspection.

In [ ]:
My predicted output:

How each return value reaches the caller:


<details>
<summary>Show answer</summary>

The lines are `Share: 4` and `Share: 3`. The loop first supplies 3, then 4, as the number of groups. The static method divides 12 by the current group count and returns that integer. Java finishes evaluating the argument to `println` before calling `println`, so the returned number becomes part of the text that is printed.

A common mistake is to treat the method call as if it prints the answer itself. This method returns a value; the caller decides to display it. That distinction will help explain what happens when a later call cannot return normally.

</details>

## Video Demonstration

Watch where the calculation stops for a zero group count. Before the command runs, predict which status lines will be skipped and which later inputs will still be processed.

<video controls preload="metadata" width="960">
  <source src="media/01_catching_and_tracing_exceptions/demo.mp4" type="video/mp4">
  <track kind="captions" src="media/01_catching_and_tracing_exceptions/captions.vtt" srclang="en" label="English">
  Your browser does not support embedded video.
</video>

[Read the catching and tracing exceptions demonstration transcript](media/01_catching_and_tracing_exceptions/transcript.md).


## Concept

### Give an unsuccessful request its own path

A campus packing team has 12 supply items. Staff want to compare independent plans that divide those items among different numbers of groups. Each input is a proposed group count, not a request to remove items from a shared stock. The calculator reports the whole number of items per group. All successful inputs used here divide the total evenly.

The initial proposals are 3 groups, 0 groups, and 4 groups. Zero groups cannot have a share. The team's rule is to report that problem, then consider the next proposal. The calculator should never invent a zero share or announce a completed calculation for the failed proposal.

An **exception** interrupts normal execution when an operation cannot complete in its usual way. Java represents the failure with an **exception object**. Its type identifies a category of failure, and its message can provide diagnostic detail. Integer division by zero produces an `ArithmeticException`; that name is a Java class, not a keyword.

A missing semicolon is different: it is a compiler error that prevents the source from being accepted for execution. Here the source is accepted, but an operation fails when the running program reaches it. This lesson uses integer division. Floating-point division by zero follows different rules.

### Leave unfinished statements and handle the failure

The **`try` keyword** introduces a block whose failures may be handled by the following handlers. The **`catch` keyword** introduces one such handler. A **handler** is the code chosen to respond to a matching exception; it does not supply a missing arithmetic result.

Read this small complete example before the larger loop:

```java
try {
    int share = 12 / 0;
    System.out.println("Share: " + share);
    System.out.println("Calculated");
} catch (ArithmeticException problem) {
    System.out.println("Cannot divide by zero.");
}
System.out.println("After handling");
```

The first statement attempts integer division. Because its divisor is zero, it does not finish assigning a value to `share`. Java leaves the `try` block, so neither of its two print statements runs. This departure before the remaining work finishes is called **abrupt completion**.

The catch header names `ArithmeticException`, which matches the failure in this example. Its variable `problem` refers to the actual exception object. The handler prints the program's chosen response, `Cannot divide by zero.` Once this handler finishes normally, the next statement prints `After handling`.

The two output lines describe recovery of control, not recovery of a quotient. Execution continues after the whole `try`/`catch` statement. It does not jump back to the failed division or replay the skipped statements. Use a specific handler for a failure you are prepared to address. An empty handler would conceal the problem from both the user and the developer.

<details class="animation-panel" open>
<summary>Follow a failed division into its handler — show or hide animation</summary>
<p><img src="media/01_catching_and_tracing_exceptions/failure_to_handler.gif" alt="The division starts; no quotient is available. The assignment does not finish. Both success messages are skipped. The handler responds to this failed operation. Execution reaches the statement after try/catch; division is not retried." width="960" style="max-width:100%;height:auto;"></p>
</details>

The failed division leaves the assignment unfinished and skips both success messages. The matching handler reports the problem. Execution then reaches After handling; it does not retry the division. This silent animation loops every 10.53 seconds. Hide it with the control above, or [view the static final state](media/01_catching_and_tracing_exceptions/failure_to_handler_still.png).

### Choose the boundary of recovery

A handler catches a failure only after Java leaves the unfinished work between the failing operation and that handler. This makes the handler's location part of the program's behavior.

For a smaller packing comparison, use 6 items and proposals for 2, 0, and 3 groups. The numbers below still represent independent alternatives. This first complete example gives each proposal its own handler:

```java
int[] divisors = {2, 0, 3};
for (int divisor : divisors) {
    try {
        System.out.println(6 / divisor);
    } catch (ArithmeticException problem) {
        System.out.println("Skipped one input");
    }
}
System.out.println("Finished");
```

The array holds the proposed group counts; `divisor` is the current count. The first iteration prints 3. The zero count enters its own handler and prints `Skipped one input`. After that handler finishes, the loop body ends normally and the loop advances to the final count. That iteration prints 2. Finally, the statement outside the loop prints `Finished`.

Now compare the same values with a handler around the entire loop:

```java
int[] divisors = {2, 0, 3};
try {
    for (int divisor : divisors) {
        System.out.println(6 / divisor);
    }
} catch (ArithmeticException problem) {
    System.out.println("Left the loop");
}
System.out.println("Finished");
```

The first division still prints 3. When zero causes a failure, Java must leave the loop to reach this outside handler. The handler prints `Left the loop`, and the next statement prints `Finished`. The final divisor, 3, is never processed. Finishing the handler does not restart the loop that was left.

Both examples handle the exception. Only the first satisfies the team's rule that later independent proposals must still be considered. An outside handler can be appropriate when the entire operation must stop after a failure; its placement should follow that requirement, not merely remove an error message.

<details class="animation-panel" open>
<summary>Compare handler placement inside and outside a loop — show or hide animation</summary>
<p><img src="media/01_catching_and_tracing_exceptions/handler_boundary_in_loop.gif" alt="Both versions start with identical independent proposals. Both versions print 3. Each version leaves its protected operation. This iteration finishes; the loop remains active. The loop has been exited; the final 3 is unvisited. Only the inside handler allows the final proposal to run." width="960" style="max-width:100%;height:auto;"></p>
</details>

Both versions compare the same proposals: divide 6 items among 2, 0, or 3 groups. With a handler inside the loop, the final proposal still runs. With the handler outside, the failure leaves the loop and the final proposal is never visited. This silent animation loops every 15.53 seconds. Hide it with the control above, or [view the static final state](media/01_catching_and_tracing_exceptions/handler_boundary_in_loop_still.png).

### Follow the failure back through its callers

A division may be hidden inside a method called by another method. To see how the same failure travels, the next complete example adds `quote`. Here a quote means a proposed items-per-group result; it is not a price.

```java
class RatioTools {
    public static int share(int total, int groups) {
        return total / groups;
    }
    public static int quote() {
        return share(12, 0);
    }
}
try {
    RatioTools.quote();
    System.out.println("Quote completed");
} catch (ArithmeticException problem) {
    System.out.println(problem.getMessage());
    System.out.println(problem.getStackTrace()[0].getMethodName());
    System.out.println(problem.getStackTrace()[1].getMethodName());
}
```

The caller enters `quote`, which calls `share` with 12 items and zero groups. The division fails inside `share`. Because that method has no matching handler, Java leaves it without returning a quotient. It also leaves the unfinished call in `quote`. The matching handler is in the outer caller, so `Quote completed` is skipped too. Leaving unfinished method calls while searching outward for a handler is called **stack unwinding**.

A **stack trace** records the chain of method calls associated with an exception. The first entry normally identifies where the exception originated; later entries identify callers. It is evidence for locating the failure, not an instruction to rerun those methods.

In the handler, `problem.getMessage()` reads the diagnostic message from the exception object. `problem.getStackTrace()` returns an array of **`StackTraceElement`** objects, each describing one recorded call. Index 0 selects the first entry, and `getMethodName()` reads the method's name from it. Index 1 selects the next entry. Both entries exist for this specific example; general diagnostic code should check the array length before indexing it.

For the course runtime, the expected lines are:

```text
/ by zero
share
quote
```

The entry order starts at `share`, where division failed, and works outward to `quote`, which called it. That is the reverse of the order in which these two methods were entered. A returned value would travel back through successful calls; here there is no quotient, and the unfinished calls are left instead.

Calling `problem.printStackTrace()` displays the full diagnostic trace. In a notebook, that trace also contains calls generated by IJava, the Java kernel. Generated file names and line numbers may change after editing or rerunning cells. Start with your own method names and the failed operation. Diagnostic message wording can also vary with the runtime, so other Java environments may display a different message.

<details class="animation-panel" open>
<summary>Follow unfinished calls and read the recorded trace — show or hide animation</summary>
<p><img src="media/01_catching_and_tracing_exceptions/unwinding_and_trace_order.gif" alt="quote is active while the caller waits. share attempts 12 / 0. No share result returns; the failed operation is recorded. quote has no local handler and cannot return a result. The recorded failure location comes before its caller, opposite to call-entry order." width="960" style="max-width:100%;height:auto;"></p>
</details>

The caller enters quote, and quote enters share. After division fails, Java leaves share and then quote to reach the caller’s handler. The recorded trace lists share before quote because it starts at the failure and works outward toward its callers. This silent animation loops every 13 seconds. Hide it with the control above, or [view the static final state](media/01_catching_and_tracing_exceptions/unwinding_and_trace_order_still.png).

## Worked Example

### Keep the calculation separate from its response

The packing team will compare three independent proposals: 3, 0, and 4 groups, each using a total of 12 items. We will keep the division in `RatioTools.share` and let the caller decide how to report success or failure.

```java
public static int share(int total, int groups) {
    return total / groups;
}
```

The two parameters receive the item total and group count for one proposal. The return expression computes items per group using integer division. If the group count is zero, evaluating that expression fails before the method can return an integer. A `return` statement does not guarantee that its expression will succeed.

### Give every proposal a chance

The array stores the three proposed group counts. An enhanced `for` loop takes them in order. Inside each iteration, `try` contains the call and the success report. Its neighboring `catch` handles a failed division for that particular proposal.

```java
try {
    System.out.println("Share: " + RatioTools.share(12, groups));
    System.out.println("Calculated");
} catch (ArithmeticException problem) {
    System.out.println("Cannot divide by zero.");
}
```

This excerpt belongs inside the loop shown in the complete program below. Java must finish the `share` call and assemble the text before it can call `println`. On a failed call, no partial `Share: ` line is printed. The next statement, which would announce `Calculated`, is also skipped. The handler supplies a failure message instead.

### Distinguish finishing an attempt from calculating a share

```java
    System.out.println("Next input");
}
System.out.println("Done");
```

In the complete program, `Next input` is after `try`/`catch` but still inside the loop. It therefore marks the end of either a successful attempt or a handled failure. The closing brace ends the loop body. `Done` follows the loop and appears once, after every proposal has had its turn.

The complete program also retains `quote`, the short method used in the earlier call-trace illustration. The loop calls `share` directly; it does not call `quote`.

In [ ]:
class RatioTools {
    public static int share(int total, int groups) {
        return total / groups;
    }
    public static int quote() {
        return share(12, 0);
    }
}
int[] groupCounts = {3, 0, 4};
for (int groups : groupCounts) {
    try {
        System.out.println("Share: " + RatioTools.share(12, groups));
        System.out.println("Calculated");
    } catch (ArithmeticException problem) {
        System.out.println("Cannot divide by zero.");
    }
    System.out.println("Next input");
}
System.out.println("Done");


Expected output:

```text
Share: 4
Calculated
Next input
Cannot divide by zero.
Next input
Share: 3
Calculated
Next input
Done
```

For the first proposal, 12 items divided among 3 groups gives 4 items per group. The call returns 4, the caller prints the share, and `Calculated` confirms success. `Next input` then marks the end of that attempt.

For zero groups, division fails inside `share`. The unfinished print and the following success message are skipped. The handler prints its failure response, then execution reaches `Next input`. That progress message means the attempt has been handled; it does not mean division succeeded.

The handler is inside the loop, so the final proposal is still visited. Four groups receive 3 items per group. After that successful attempt, the loop ends and `Done` prints once. The 12-item total stays the same for each proposal because these are alternative plans.

## Guided Practice

### Follow a failure before successful requests

The packing team is still comparing independent ways to divide 12 items. The next program changes only the group counts to `{0, 6, 2}`. Before running it, predict every output line in order. Include the success, failure, progress, and final messages. Explain what happens to the first attempt to print a share when the division fails.

In [ ]:
My predicted complete output:

What interrupts the first share message:


In [ ]:
class RatioTools {
    public static int share(int total, int groups) {
        return total / groups;
    }
    public static int quote() {
        return share(12, 0);
    }
}
int[] groupCounts = {0, 6, 2};
for (int groups : groupCounts) {
    try {
        System.out.println("Share: " + RatioTools.share(12, groups));
        System.out.println("Calculated");
    } catch (ArithmeticException problem) {
        System.out.println("Cannot divide by zero.");
    }
    System.out.println("Next input");
}
System.out.println("Done");


Run the program above once. Record its complete output below and compare it with your prediction. If they differ, identify the first differing line and explain the control-flow decision that caused it.

In [ ]:
My actual output:

First difference, or why my prediction matched:


Now connect those printed lines to the statements that produced them. For each group count, record whether the division returns, whether `Calculated` prints, whether the handler runs, and whether `Next input` prints. Identify the exception type and what the variable `problem` refers to. Finally, name the first statement reached after the handler completes.

In [ ]:
groups = 0: division returns / Calculated / handler / Next input:

groups = 6: division returns / Calculated / handler / Next input:

groups = 2: division returns / Calculated / handler / Next input:

Exception type and the object referred to by problem:

First statement after the handler:


<details>
<summary>Show answer</summary>

The first input is 0. Integer division fails inside share before it returns a result. The surrounding println cannot finish its argument, so it prints no Share line. Calculated is also skipped. The matching catch prints Cannot divide by zero. and then control reaches Next input. With 6 groups, share returns 2; with 2 groups, it returns 6. Both successful iterations print Share, Calculated and Next input in that order. The catch is inside the loop, so the first failure does not discard either later input. Done prints once after all three iterations. Handling the exception does not resume the failed division or the skipped statements inside try. The exception type named by the handler is ArithmeticException. The catch variable problem refers to the exception object raised during this failed division; the fixed printed sentence is the program’s chosen response. It is not a replacement numeric result. In this loop, the failed share call leaves the current print unfinished and Java finds the catch for that iteration. The separate method-call check below makes the unfinished calls and their recorded order visible.

```java
class RatioTools {
    public static int share(int total, int groups) {
        return total / groups;
    }
    public static int quote() {
        return share(12, 0);
    }
}
int[] groupCounts = {0, 6, 2};
for (int groups : groupCounts) {
    try {
        System.out.println("Share: " + RatioTools.share(12, groups));
        System.out.println("Calculated");
    } catch (ArithmeticException problem) {
        System.out.println("Cannot divide by zero.");
    }
    System.out.println("Next input");
}
System.out.println("Done");
```

Expected output:

```text
Cannot divide by zero.
Next input
Share: 2
Calculated
Next input
Share: 6
Calculated
Next input
Done
```

Common error: Printing Share: 0 for a failed division. Printing Calculated after the failed call. Skipping the two later inputs even though the handler is inside the loop. Using the earlier worked example’s group counts or output.

</details>

### Trace a failure through two method calls

A packing request now asks `quote` to calculate a share of 18 items. The method calls `share`, so the request has two active method calls when division begins. The program below prints messages on entry and before normal return. Its first request uses zero groups. Before running, predict the complete output, including the two method names taken from the exception’s stack trace. List any entry, leave, or quote-result messages that will be skipped.

In [ ]:
Predicted zero-group output:

Skipped messages and why:

First two recorded method names:


In [ ]:
class TracePacking {
    public static int share(int total, int groups) {
        System.out.println("Enter share");
        int result = total / groups;
        System.out.println("Leave share");
        return result;
    }
    public static int quote(int groups) {
        System.out.println("Enter quote");
        int result = share(18, groups);
        System.out.println("Leave quote");
        return result;
    }
}
try {
    System.out.println("Start request");
    int result = TracePacking.quote(0);
    System.out.println("Quote: " + result);
} catch (ArithmeticException problem) {
    System.out.println("Problem: " + problem.getMessage());
    StackTraceElement[] calls = problem.getStackTrace();
    System.out.println("Top method: " + calls[0].getMethodName());
    System.out.println("Caller method: " + calls[1].getMethodName());
}
System.out.println("After handling");


Run the zero-group request. Record the output, then explain why an `Enter` message can appear without the corresponding `Leave` message. Distinguish the order in which the methods were called from the order in which their names appear in the recorded stack trace.

In [ ]:
Actual failure output:

Why the leave messages are absent:

Call order compared with recorded trace order:


The next program changes the request to three groups while keeping the same methods and messages. Predict the complete output before running it. Explain which messages can now appear because `share` and `quote` return normally.

In [ ]:
Predicted three-group output:

Messages enabled by normal returns:


In [ ]:
class TracePacking {
    public static int share(int total, int groups) {
        System.out.println("Enter share");
        int result = total / groups;
        System.out.println("Leave share");
        return result;
    }
    public static int quote(int groups) {
        System.out.println("Enter quote");
        int result = share(18, groups);
        System.out.println("Leave quote");
        return result;
    }
}
try {
    System.out.println("Start request");
    int result = TracePacking.quote(3);
    System.out.println("Quote: " + result);
} catch (ArithmeticException problem) {
    System.out.println("Problem: " + problem.getMessage());
    StackTraceElement[] calls = problem.getStackTrace();
    System.out.println("Top method: " + calls[0].getMethodName());
    System.out.println("Caller method: " + calls[1].getMethodName());
}
System.out.println("After handling");


Run the three-group request and record the result. Compare it with the zero-group request: which path reaches the caller with a return value, and which path reaches the handler with an exception object? Explain why `After handling` appears in both programs.

In [ ]:
Actual success output:

How the successful and failed calls reach different paths:

Why After handling appears in both:


<details>
<summary>Show answer</summary>

The caller first prints Start request. quote prints Enter quote and calls share, which prints Enter share. Division by 0 then raises ArithmeticException. The remaining Leave share message and return are skipped; quote also cannot finish its call, so Leave quote and its return are skipped. The caller cannot initialize result or print Quote. Java finds the caller’s matching catch. problem refers to the actual exception object; getMessage returns / by zero in the selected Workspace runtime. getStackTrace returns recorded entries with share first and quote next: the failing method followed by its caller. That recorded order is opposite the two Enter messages, which showed calls being entered. The handler does not resume either method. After handling runs after the whole try/catch. In the comparison, 18 / 3 returns 6, both Leave messages run in return order, Quote: 6 prints, and the catch body is skipped.

```java
class TracePacking {
    public static int share(int total, int groups) {
        System.out.println("Enter share");
        int result = total / groups;
        System.out.println("Leave share");
        return result;
    }
    public static int quote(int groups) {
        System.out.println("Enter quote");
        int result = share(18, groups);
        System.out.println("Leave quote");
        return result;
    }
}
try {
    System.out.println("Start request");
    int result = TracePacking.quote(0);
    System.out.println("Quote: " + result);
} catch (ArithmeticException problem) {
    System.out.println("Problem: " + problem.getMessage());
    StackTraceElement[] calls = problem.getStackTrace();
    System.out.println("Top method: " + calls[0].getMethodName());
    System.out.println("Caller method: " + calls[1].getMethodName());
}
System.out.println("After handling");
```

Expected output:

```text
Start request
Enter quote
Enter share
Problem: / by zero
Top method: share
Caller method: quote
After handling
```

Common error: Reading the Enter order as the exception trace order. Printing Leave share or Leave quote after the division fails. Assuming a matching catch repairs and resumes the unfinished methods. Comparing generated notebook file names or line numbers instead of these two method names. Assuming the label After handling means the catch ran on the successful path.

**Check case 2.** A valid divisor allows both method calls to return. Both Leave messages and Quote appear; the handler’s message and trace lines do not. After handling still runs because it follows the whole try/catch.

```java
class TracePacking {
    public static int share(int total, int groups) {
        System.out.println("Enter share");
        int result = total / groups;
        System.out.println("Leave share");
        return result;
    }
    public static int quote(int groups) {
        System.out.println("Enter quote");
        int result = share(18, groups);
        System.out.println("Leave quote");
        return result;
    }
}
try {
    System.out.println("Start request");
    int result = TracePacking.quote(3);
    System.out.println("Quote: " + result);
} catch (ArithmeticException problem) {
    System.out.println("Problem: " + problem.getMessage());
    StackTraceElement[] calls = problem.getStackTrace();
    System.out.println("Top method: " + calls[0].getMethodName());
    System.out.println("Caller method: " + calls[1].getMethodName());
}
System.out.println("After handling");
```

Expected output:

```text
Start request
Enter quote
Enter share
Leave share
Leave quote
Quote: 6
After handling
```

</details>

### Complete a handler for each box count

A shipping assistant is comparing plans for placing 9 parcels into boxes. Each box count is a separate proposal. Zero boxes cannot receive the parcels, but that failed proposal should not prevent checking three boxes.

The draft below is incomplete and is for reading only. Choose replacements for `PROTECTED_WORK`, `HANDLE`, and `FAILURE` from `try`, `catch`, and `ArithmeticException`, using each once. Record your choices and predict every output line before copying the draft into the Java work cell. Keep all other statements unchanged.

```java
class PackShare {
    public static int perBox(int parcels, int boxes) {
        return parcels / boxes;
    }
}
int[] boxCounts = {0, 3};
for (int boxes : boxCounts) {
    PROTECTED_WORK {
        int each = PackShare.perBox(9, boxes);
        System.out.println("Each: " + each);
        System.out.println("Ready");
    } HANDLE (FAILURE problem) {
        System.out.println("Needs a box count.");
    }
    System.out.println("Checked");
}
System.out.println("Complete");
```

In [ ]:
PROTECTED_WORK replacement:
HANDLE replacement:
FAILURE replacement:

Predicted completed-program output:


Run your completed program. Record the complete output and identify the operation that fails for zero boxes. Explain what `problem` refers to in that handler, why `Ready` prints for only one proposal, and why `Checked` prints for both.

In [ ]:
Actual output:

Failing operation and exception object:

Why Ready and Checked have different counts:


<details>
<summary>Show answer</summary>

PROTECTED_WORK is try, HANDLE is catch and FAILURE is ArithmeticException. For box count 0, the call to perBox cannot return a result, so each is not initialized and neither Each nor Ready is printed. The catch prints Needs a box count. Checked runs after that handler. For box count 3, integer division returns 3, so Each: 3 and Ready print; catch is skipped and Checked still prints. Complete appears once after the loop. The catch is inside the loop and receives the exception object for its failed calculation.

```java
class PackShare {
    public static int perBox(int parcels, int boxes) {
        return parcels / boxes;
    }
}
int[] boxCounts = {0, 3};
for (int boxes : boxCounts) {
    try {
        int each = PackShare.perBox(9, boxes);
        System.out.println("Each: " + each);
        System.out.println("Ready");
    } catch (ArithmeticException problem) {
        System.out.println("Needs a box count.");
    }
    System.out.println("Checked");
}
System.out.println("Complete");
```

Expected output:

```text
Needs a box count.
Checked
Each: 3
Ready
Checked
Complete
```

Common error: Replacing catch with another loop. Placing Ready after the catch so it falsely reports success after failure. Moving Checked into the successful try body. Changing the provided inputs to avoid handling the failing case.

</details>

### Make the failure report more useful

The packing team needs to know which group count failed, along with the exception’s diagnostic message. In the next program, keep `RatioTools`, the `{0, 6, 2}` inputs, and every success and progress message unchanged. Replace only the print statement inside `catch` with:

```java
System.out.println("Groups " + groups + ": " + problem.getMessage());
```

Before editing and running, predict the full output. Identify which part of the new report comes from the current input and which part comes from the exception object.

In [ ]:
Predicted output with the new report:

Part supplied by the input:
Part supplied by the exception object:


In [ ]:
class RatioTools {
    public static int share(int total, int groups) {
        return total / groups;
    }
    public static int quote() {
        return share(12, 0);
    }
}
int[] groupCounts = {0, 6, 2};
for (int groups : groupCounts) {
    try {
        System.out.println("Share: " + RatioTools.share(12, groups));
        System.out.println("Calculated");
    } catch (ArithmeticException problem) {
        System.out.println("Cannot divide by zero.");
    }
    System.out.println("Next input");
}
System.out.println("Done");


Make the one-line change and run the program. Record the complete output. Compare the new diagnostic with your prediction and explain any difference without changing the successful calculation or its messages.

In [ ]:
Actual baseline output:

Diagnostic comparison and explanation:


Next, test whether separate failures receive separate responses. Change only the array initializer in the program above to `{2, 0, 0}`. Before running it again, predict every output line and the number of times the handler will run.

In [ ]:
Predicted repeated-failure output:

Predicted handler count and reason:


Run the `{2, 0, 0}` version and record its output. Explain why the first zero does or does not prevent processing the second zero. Then restore `{0, 6, 2}`, rerun, and record the restored result so the program is ready for later use.

In [ ]:
Actual repeated-failure output:

How execution reaches the second zero:

Actual output after restoring the baseline:


<details>
<summary>Show answer</summary>

The handler now combines the current input groups with the actual exception message. For zero groups it prints Groups 0: / by zero in this Workspace runtime. That message does not provide a division result, so there is still no Share or Calculated line for that input. All Next input lines remain because their statement follows try/catch inside the loop. The valid inputs still produce shares 2 and 6. With {2, 0, 0}, the first calculation returns 6 and each later zero produces its own caught exception and diagnostic. The repeated failure does not stop the loop or remove the final Done.

```java
class RatioTools {
    public static int share(int total, int groups) {
        return total / groups;
    }
    public static int quote() {
        return share(12, 0);
    }
}
int[] groupCounts = {0, 6, 2};
for (int groups : groupCounts) {
    try {
        System.out.println("Share: " + RatioTools.share(12, groups));
        System.out.println("Calculated");
    } catch (ArithmeticException problem) {
        System.out.println("Groups " + groups + ": " + problem.getMessage());
    }
    System.out.println("Next input");
}
System.out.println("Done");
```

Expected output:

```text
Groups 0: / by zero
Next input
Share: 2
Calculated
Next input
Share: 6
Calculated
Next input
Done
```

Common error: Using the fixed friendly sentence while claiming to display the exception’s message. Moving Next input into catch and losing it on successful inputs. Treating the diagnostic as a successful calculated value. Handling only the first of two failed inputs.

**Additional test: `One valid input followed by two zero group counts`.** The per-input handler runs twice. Next input runs three times, and Done still runs once after the loop.

```java
class RatioTools {
    public static int share(int total, int groups) {
        return total / groups;
    }
    public static int quote() {
        return share(12, 0);
    }
}
int[] groupCounts = {2, 0, 0};
for (int groups : groupCounts) {
    try {
        System.out.println("Share: " + RatioTools.share(12, groups));
        System.out.println("Calculated");
    } catch (ArithmeticException problem) {
        System.out.println("Groups " + groups + ": " + problem.getMessage());
    }
    System.out.println("Next input");
}
System.out.println("Done");
```

Expected output:

```text
Share: 6
Calculated
Next input
Groups 0: / by zero
Next input
Groups 0: / by zero
Next input
Done
```

</details>

### Repair a handler that stops the remaining proposals

The team must check every proposed group count. The draft below catches a failure, but places the entire loop inside `try`. For `{6, 0, 3}`, predict its complete output and identify the proposal it never reaches. Keep this faulty draft in the reading cell.

Plan a repair that puts `try` and `catch` inside the loop. Place `Next input` after that handler within the loop, and leave `Done` after the loop. Preserve the methods, inputs, and all message text. Predict the repaired output before writing the complete repaired program in the Java work cell.

```java
class RatioTools {
    public static int share(int total, int groups) {
        return total / groups;
    }
    public static int quote() {
        return share(12, 0);
    }
}
int[] groupCounts = {6, 0, 3};
try {
    for (int groups : groupCounts) {
        System.out.println("Share: " + RatioTools.share(12, groups));
        System.out.println("Calculated");
        System.out.println("Next input");
    }
} catch (ArithmeticException problem) {
    System.out.println("Cannot divide by zero.");
}
System.out.println("Done");
```

In [ ]:
Predicted faulty output:

Skipped proposal and missing progress messages:

My handler-placement repair:

Predicted repaired output:


Run the repaired program and record all output. Explain how the new handler location allows the final group count to be processed. Why would handling the exception outside the loop fail to restart that loop where it stopped?

In [ ]:
Actual repaired output:

Why the final proposal is reached:

Why the outside handler cannot resume the exited loop:


<details>
<summary>Show answer</summary>

With the handler outside, the first input 6 returns share 2 and prints its two progress messages. The zero input causes the method call and then the whole loop to end abruptly while Java searches for the outside catch. That input never reaches Next input, and the later input 3 is not visited. After the outside catch, control reaches Done; it does not reenter the loop. The repair moves the handler inside each iteration and places Next input after that handler. The zero input now receives its response and progress marker, and the later input 3 returns share 4. No method or input change is needed.

```java
class RatioTools {
    public static int share(int total, int groups) {
        return total / groups;
    }
    public static int quote() {
        return share(12, 0);
    }
}
int[] groupCounts = {6, 0, 3};
for (int groups : groupCounts) {
    try {
        System.out.println("Share: " + RatioTools.share(12, groups));
        System.out.println("Calculated");
    } catch (ArithmeticException problem) {
        System.out.println("Cannot divide by zero.");
    }
    System.out.println("Next input");
}
System.out.println("Done");
```

Expected output:

```text
Share: 2
Calculated
Next input
Cannot divide by zero.
Next input
Share: 4
Calculated
Next input
Done
```

Common error: Changing zero to a valid number instead of repairing handler placement. Keeping the catch outside and expecting the loop to restart afterward. Leaving Next input inside try so it is still skipped on failure. Printing Calculated from catch and falsely reporting success.

</details>

## Independent Practice

### Build a packing calculation that continues after failure

A shipping coordinator is comparing ways to distribute 10 parcels among containers. Each proposed container count describes a separate plan; the program does not remove parcels as it checks a plan. The initial proposals are `{2, 0, 5}`. Valid proposals report whole parcels per container. An invalid zero count must receive a message without preventing later proposals.

Write a complete `ParcelMath` class with `public static int perContainer(int parcels, int containers)` returning `parcels / containers`. Use an enhanced `for` loop over `int[] containerCounts = {2, 0, 5};`. Inside each iteration, attempt the calculation for 10 parcels. On success, print `Per container: ` plus the result, followed by `Packed`. Catch `ArithmeticException` and print `Choose a nonzero container count.` After either path, print `Next input` within the loop. Print `Finished` once after the loop.

Before writing the program, outline where the calculation and handler belong and predict the complete baseline output.

In [ ]:
My calculation method and handler placement:

Predicted baseline output:


Run your program with `{2, 0, 5}`. Record the complete output and explain which success statements the zero proposal skips. Identify where control goes after its handler and why the last valid proposal is still processed.

In [ ]:
Actual baseline output:

Statements skipped for zero:

Continuation after the handler:

Why the final valid proposal is processed:


### Check empty input and different failure positions

A program that works for one array may still put its handler in the wrong place. Test these five arrays by changing only `containerCounts`: `{2, 0, 5}`, `{}`, `{0, 2}`, `{5, 0}`, and `{0, 0}`. Before running the tests, predict every output line for each case. For each prediction, also count the `Per container`, `Packed`, `Next input`, and `Finished` lines.

In [ ]:
Predicted output and four message counts for {2, 0, 5}:

Predicted output and four message counts for {}:

Predicted output and four message counts for {0, 2}:

Predicted output and four message counts for {5, 0}:

Predicted output and four message counts for {0, 0}:


Run each of the five cases and record the actual output and message counts. Explain what the empty array checks, how an initial zero differs from a final zero, and why repeated zeros should receive separate responses. Identify a case that would expose a handler placed around the whole loop. If you find a mismatch, repair the program and repeat the affected tests. Restore `{2, 0, 5}` and record one final run.

In [ ]:
Actual output and counts for {2, 0, 5}:

Actual output and counts for {}:

Actual output and counts for {0, 2}:

Actual output and counts for {5, 0}:

Actual output and counts for {0, 0}:

What the empty, initial-zero, final-zero, and repeated-zero cases check:

Why Finished appears once in each case:

Case that exposes an outside-loop handler, with reason:

Corrections and repeated checks, or why none were needed:

Actual output after restoring the baseline:


<details>
<summary>Show answer</summary>

ParcelMath.perContainer performs integer division. The first input 2 returns 5, so Per container: 5 and Packed print before Next input. The zero input fails before the Per container print can finish its argument; Packed is skipped and the matching catch prints Choose a nonzero container count. Next input follows either path. The final input 5 returns 2 and prints the two success lines and progress marker. Finished runs after the loop. The class and all caller data are included, and the handler is inside each iteration so one invalid count does not discard later inputs. The empty array performs no loop work and prints only Finished. With {0, 2}, handling the first zero lets the valid second input produce 5 and Packed. With {5, 0}, the final caught failure still reaches Next input and then Finished. With {0, 0}, both failures get a response and progress marker, and no Packed line appears. These tests check handler placement and progress rather than only one successful quotient. An outside catch would exit the loop on its first failure and skip any later inputs.

```java
class ParcelMath {
    public static int perContainer(int parcels, int containers) {
        return parcels / containers;
    }
}
int[] containerCounts = {2, 0, 5};
for (int containers : containerCounts) {
    try {
        System.out.println("Per container: " + ParcelMath.perContainer(10, containers));
        System.out.println("Packed");
    } catch (ArithmeticException problem) {
        System.out.println("Choose a nonzero container count.");
    }
    System.out.println("Next input");
}
System.out.println("Finished");
```

Expected output:

```text
Per container: 5
Packed
Next input
Choose a nonzero container count.
Next input
Per container: 2
Packed
Next input
Finished
```

Common error: Placing the handler around the entire loop and losing later inputs. Printing Packed from a failed path. Returning a made-up numeric result for division by zero. Putting Next input only in the successful try body. Changing fixed method names, input counts or output labels.

**Additional test: Empty input array.** With no inputs, there is no loop iteration, so there is no calculation, handler response or Next input. Finished still prints once.

```java
class ParcelMath {
    public static int perContainer(int parcels, int containers) {
        return parcels / containers;
    }
}
int[] containerCounts = {};
for (int containers : containerCounts) {
    try {
        System.out.println("Per container: " + ParcelMath.perContainer(10, containers));
        System.out.println("Packed");
    } catch (ArithmeticException problem) {
        System.out.println("Choose a nonzero container count.");
    }
    System.out.println("Next input");
}
System.out.println("Finished");
```

Expected output:

```text
Finished
```

**Additional test: Initial zero followed by a valid count.** The first failure receives its response and Next input. The valid second count still returns 5 and reaches Packed.

```java
class ParcelMath {
    public static int perContainer(int parcels, int containers) {
        return parcels / containers;
    }
}
int[] containerCounts = {0, 2};
for (int containers : containerCounts) {
    try {
        System.out.println("Per container: " + ParcelMath.perContainer(10, containers));
        System.out.println("Packed");
    } catch (ArithmeticException problem) {
        System.out.println("Choose a nonzero container count.");
    }
    System.out.println("Next input");
}
System.out.println("Finished");
```

Expected output:

```text
Choose a nonzero container count.
Next input
Per container: 5
Packed
Next input
Finished
```

**Additional test: A valid count followed by a final zero.** The first count returns 2. The final failure still reaches its Next input, and Finished remains reachable after the loop.

```java
class ParcelMath {
    public static int perContainer(int parcels, int containers) {
        return parcels / containers;
    }
}
int[] containerCounts = {5, 0};
for (int containers : containerCounts) {
    try {
        System.out.println("Per container: " + ParcelMath.perContainer(10, containers));
        System.out.println("Packed");
    } catch (ArithmeticException problem) {
        System.out.println("Choose a nonzero container count.");
    }
    System.out.println("Next input");
}
System.out.println("Finished");
```

Expected output:

```text
Per container: 2
Packed
Next input
Choose a nonzero container count.
Next input
Finished
```

**Additional test: Two consecutive zero counts.** Each failed input has its own response and progress marker. Neither reaches Packed, but the program still prints Finished once.

```java
class ParcelMath {
    public static int perContainer(int parcels, int containers) {
        return parcels / containers;
    }
}
int[] containerCounts = {0, 0};
for (int containers : containerCounts) {
    try {
        System.out.println("Per container: " + ParcelMath.perContainer(10, containers));
        System.out.println("Packed");
    } catch (ArithmeticException problem) {
        System.out.println("Choose a nonzero container count.");
    }
    System.out.println("Next input");
}
System.out.println("Finished");
```

Expected output:

```text
Choose a nonzero container count.
Next input
Choose a nonzero container count.
Next input
Finished
```

</details>

## Summary

An exception interrupts an operation that cannot complete normally. A matching `catch` responds after Java leaves the unfinished work. Handling a failure does not retry that work or create a missing return value.

A handler inside a loop can let later independent inputs run. A handler outside the loop is reached only after the loop has been left. When a failure leaves method calls, its stack trace helps locate the failed operation and its callers.

### Retrieve the main idea

Without reopening the worked example, explain why completing a `catch` block does not retry the failed division. For a caller that invokes `quote`, which invokes `share`, describe how an exception from `share` reaches a handler in the caller. State which of those method names appears first in the exception’s recorded stack trace and why.

In [ ]:
What happens after catch, and why the division is not retried:

Failure path from share through quote to the caller:

First recorded method name and reason:


<details>
<summary>Show answer</summary>

A matching handler runs after the failed division and unfinished statements have been left. When that handler finishes normally, execution continues after the `try`/`catch`, so the division is not retried.

The caller enters `quote`, which enters `share`. A failure in `share` leaves that call and the unfinished call in `quote` until the caller’s matching handler is reached. For this example, the recorded method names start with `share`, then `quote`: the failure location comes before its caller. Do not confuse that diagnostic order with the order of method entry.

</details>

## Reflection

Imagine a staff member submitting several independent form entries. Describe one failure for which the application should report a problem with one entry and continue with later entries. Then describe a failure that makes continuing the whole submission inappropriate. Explain how these different requirements would affect the scope of the protected operation and its handler.

In [ ]:
One entry-specific failure and why later entries can continue:

One failure that should stop the submission and why:

How the requirements change handler placement:


### Looking Ahead

You have handled an exception produced by division and traced the calls it interrupted. In the next lesson, you will define a method’s own failure rules and deliberately throw an exception when an input breaks those rules.

## Supplemental Reading

- [Catching and handling Java exceptions](https://dev.java/learn/exceptions/catching-handling/) explains handler placement and recovery.
- [Java 21 Throwable API](https://docs.oracle.com/en/java/javase/21/docs/api/java.base/java/lang/Throwable.html) documents messages and stack-trace inspection.
- [Java 21 exception rules](https://docs.oracle.com/javase/specs/jls/se21/html/jls-11.html) defines exception control flow.
